# Bridges on your own DICOM data (pyRadPlan **or** QIB → PortPy)

An end-to-end, **fill-in-the-blanks** workflow that mirrors the real cervix /
rectum VMAT case run during development:

> DICOM → pyCERR → influence matrix (**pyRadPlan** *or* **QIB**) → PortPy
> dataset → PortPy optimization with your constraints → dose back in `planC`
> → DVH comparison against the clinical dose.

Edit the **CONFIG** cell, pick a **dose engine**, set your **prescription /
OAR constraints**, then Run-All.

**Requires:** `pip install portpy` (always) and, for the pyRadPlan engine,
`pip install pyRadPlan`.

## 1. CONFIG — plug in your dataset

Set the DICOM directory, the planning scan, and the structure names as they
appear in your RTSTRUCT. All placeholders below are marked `# <-- EDIT`.

In [ ]:
# ----------------------------- CONFIG ------------------------------------
DICOM_DIR   = r"C:/path/to/your/DICOM/folder"   # <-- EDIT: CT + RTSTRUCT (+ RTPLAN, RTDOSE)
SCAN_NUM    = 0            # <-- EDIT: index of the planning CT in planC.scan
TARGET_NAME = "PTV"       # <-- EDIT: target structure name in the RTSTRUCT
BODY_NAME   = "BODY"      # <-- EDIT: external/BODY structure name (calc box)

# Beam geometry for the re-plan (VMAT arcs are NOT reproduced; use static IMRT
# fields). Set GANTRY_ANGLES = None to read treatment beams from planC.beams[0].
GANTRY_ANGLES = [0.0, 51.0, 102.0, 153.0, 204.0, 255.0, 306.0]   # <-- EDIT

# Prescription
PRESCRIPTION_GY = 25.0     # <-- EDIT: total prescription dose (Gy)
NUM_FRACTIONS   = 5        # <-- EDIT

# OAR objectives: {structure_name: overdose_limit_gy}
OAR_OBJECTIVES = {         # <-- EDIT to match your OARs
    "Rectum":  18.0,
    "Bladder": 18.0,
}

# Dose engine: "pyradplan" (physical pencil beam) or "qib" (pyCERR native)
DOSE_ENGINE = "pyradplan"  # <-- EDIT: "pyradplan" or "qib"

# Beamlet / grid resolution (multiples of 2.5 mm for PortPy)
BIXEL_WIDTH_MM = 5.0
DOSE_GRID_MM   = 5.0
SOLVER         = "SCS"     # free; use "MOSEK"/"GUROBI" if licensed

# Clinical dose already in the dataset? (RTDOSE) -> DVH comparison at the end
COMPARE_TO_CLINICAL = True
CLINICAL_DOSE_NUM   = 0    # index into planC.dose of the clinical RTDOSE
# -------------------------------------------------------------------------

## 2. Load the DICOM dataset

In [ ]:
import time, tempfile, numpy as np
from cerr import plan_container as pc
from cerr.imrtp import portpy_bridge as ppb

t0 = time.time()
planC = pc.loadDcmDir(DICOM_DIR)
print("loaded in %.1fs" % (time.time() - t0))
print("scans:", [(i, s.scanType, tuple(s.getScanSize()))
                 for i, s in enumerate(planC.scan)])
print("structures:", [s.structureName for s in planC.structure])
print("doses:", len(planC.dose), "| beams:", len(planC.beams))

def sidx(name):
    hits = [i for i, s in enumerate(planC.structure)
            if s.structureName == name]
    if not hits:
        raise ValueError("structure %r not found; available: %s"
                         % (name, [s.structureName for s in planC.structure]))
    return hits[0]

target = sidx(TARGET_NAME)
bodyN  = sidx(BODY_NAME)
oar_nums = {nm: sidx(nm) for nm in OAR_OBJECTIVES}
struct_nums = [bodyN, target] + list(oar_nums.values())
print("target=%d body=%d oars=%s" % (target, bodyN, oar_nums))

## 3. (Optional) DVH helper + clinical baseline

If the dataset ships an RTDOSE, we tabulate its DVH for comparison later.

In [ ]:
from cerr import dvh

def dvh_stats(structNum, doseNum):
    d, v, _ = dvh.getDVH(structNum, doseNum, planC)
    d = np.asarray(d); v = np.asarray(v)
    o = np.argsort(d)[::-1]; d, v = d[o], v[o]
    cum = np.cumsum(v); tot = cum[-1]
    Dp = lambda p: float(np.interp(p / 100.0 * tot, cum, d))
    return {"mean": round(float((d * v).sum() / tot), 2),
            "max": round(float(d.max()), 2),
            "D95": round(Dp(95), 2), "D2": round(Dp(2), 2)}

if COMPARE_TO_CLINICAL and len(planC.dose) > CLINICAL_DOSE_NUM:
    print("--- CLINICAL dose[%d] ---" % CLINICAL_DOSE_NUM)
    for nm in [TARGET_NAME] + list(OAR_OBJECTIVES):
        print("  %-12s" % nm, dvh_stats(sidx(nm), CLINICAL_DOSE_NUM))
else:
    print("no clinical dose to compare against")

## 4. Build the influence matrix + PortPy dataset (engine of your choice)

`DOSE_ENGINE = "pyradplan"` uses pyRadPlan's pencil-beam engine;
`"qib"` uses pyCERR's native QIB engine (no pyRadPlan needed).

In [ ]:
data_dir = tempfile.mkdtemp()
patient_id = "Case_1"

if DOSE_ENGINE == "pyradplan":
    from cerr.imrtp import pyradplan_bridge as prp
    iso = prp.targetCentroidMm(planC, [target])
    ct, cst, pln = prp.planFromPlanC(
        planC, scanNum=SCAN_NUM, beamsNum=None,
        objectives={target: [prp.squaredDeviation(PRESCRIPTION_GY)]},
        structNums=struct_nums, targetStructNums=[target],
        gantryAngles=GANTRY_ANGLES, isoCenter=iso,
        bixelWidth=BIXEL_WIDTH_MM,
        doseGridResolution={"x": DOSE_GRID_MM, "y": DOSE_GRID_MM,
                            "z": DOSE_GRID_MM})
    t0 = time.time()
    stf, dij = prp.calcDoseInfluence(ct, cst, pln)
    print("pyRadPlan influence %.1fs" % (time.time() - t0))
    ppb.writePortpyFromPyRadPlan(planC, dij, stf, outDir=data_dir,
                                 patientId=patient_id, scanNum=SCAN_NUM,
                                 structNums=struct_nums, bodyStructNum=bodyN,
                                 isoCenter=iso)

elif DOSE_ENGINE == "qib":
    from cerr.imrtp import imrtp_problem as imp
    from cerr.imrtp.dosecalc import generateQIBInfluence
    im = imp.initIMRTProblem(planC)
    g = imp.addGoal(im, target, planC); g.isTarget = "yes"; g.xySampleRate = 2
    if GANTRY_ANGLES:
        imp.addEquispacedBeams(im, len(GANTRY_ANGLES), GANTRY_ANGLES[0], planC)
        for b, ang in zip(im.beams, GANTRY_ANGLES):
            b.gantryAngle = float(ang)
    else:
        imp.addEquispacedBeams(im, 7, 0.0, planC)
    im.params.algorithm = "QIB"
    for b in im.beams:
        imp.conditionBeam(b, im, planC)
    t0 = time.time()
    generateQIBInfluence(im, planC)
    print("QIB influence %.1fs" % (time.time() - t0))
    ppb.writePortpyFromQIB(planC, im, outDir=data_dir, patientId=patient_id,
                           scanNum=SCAN_NUM, structNums=struct_nums,
                           bodyStructNum=bodyN)
else:
    raise ValueError("DOSE_ENGINE must be 'pyradplan' or 'qib'")

print("PortPy dataset ->", data_dir)

## 5. Optimize with your constraints and import the dose

`optimizeAndImport` maps `PRESCRIPTION_GY` + `OAR_OBJECTIVES` into PortPy's
clinical criteria, runs the CVXPy solve, and appends the dose to `planC.dose`.

In [ ]:
t0 = time.time()
sol, doseNum, planC = ppb.optimizeAndImport(
    planC, dataDir=data_dir, patientId=patient_id,
    prescriptionGy=PRESCRIPTION_GY, numFractions=NUM_FRACTIONS,
    scanNum=SCAN_NUM, targetName=TARGET_NAME,
    oarObjectives=OAR_OBJECTIVES, solver=SOLVER)
print("optimized + imported in %.1fs -> planC.dose[%d]"
      % (time.time() - t0, doseNum))

## 6. Compare the optimized plan to the clinical dose

In [ ]:
print("--- PortPy optimized dose[%d] (%s engine) ---" % (doseNum, DOSE_ENGINE))
for nm in [TARGET_NAME] + list(OAR_OBJECTIVES):
    print("  %-12s" % nm, dvh_stats(sidx(nm), doseNum))

if COMPARE_TO_CLINICAL and len(planC.dose) > CLINICAL_DOSE_NUM + 1:
    print("\n--- CLINICAL dose[%d] (reference) ---" % CLINICAL_DOSE_NUM)
    for nm in [TARGET_NAME] + list(OAR_OBJECTIVES):
        print("  %-12s" % nm, dvh_stats(sidx(nm), CLINICAL_DOSE_NUM))

In [ ]:
import matplotlib.pyplot as plt
scan3M = planC.scan[SCAN_NUM].getScanArray()
dose3M = planC.dose[doseNum].doseArray
k = scan3M.shape[2] // 2
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(scan3M[:, :, k], cmap="gray")
m = ax.imshow(np.ma.masked_less(dose3M[:, :, k], 0.1 * dose3M.max()),
              cmap="jet", alpha=0.5)
plt.colorbar(m, ax=ax, label="Gy", fraction=0.046)
ax.set_title("optimized dose (%s), slice %d" % (DOSE_ENGINE, k)); ax.axis("off")
plt.show()

## Caveats (same as the development run)

- **VMAT arcs are not reproduced.** This is a static-field IMRT re-plan; a
  clinical dual-arc VMAT plan will generally show tighter target coverage. Use
  more fields / ring structures / DVH criteria to close the gap.
- **Runtime.** Many beamlets × the free `SCS` solver can take many minutes on a
  full-resolution clinical case. Coarsen `BIXEL_WIDTH_MM` / `DOSE_GRID_MM`, or
  use a licensed QP solver (`MOSEK`, `GUROBI`) for speed.
- **Orientation & grids** are handled automatically (LPS reorientation; dose
  resampled back onto the scan grid), including scans with a flipped
  `imageOrientationPatient`.
- **Prescription scaling.** `PortPy` optimizes per-fraction dose; the imported
  `planC.dose` is scaled to the **total** prescription (`× NUM_FRACTIONS`).